# Logistic Regression Analysis for Heart Disease Prediction
**Machine Learning Assignment — Member 1**  
**Algorithm: Logistic Regression (Binary Classification)**

---

## 1. Problem Definition

Predict whether a patient has heart disease using the Cleveland dataset. The model will classify patients into:
- **0** = no heart disease
- **1** = heart disease present

The goal is to build a reproducible logistic regression pipeline with clean preprocessing, model training, and evaluation.

## 2. Dataset Description

The Cleveland dataset contains clinical measurements for heart disease diagnosis. It has 14 columns: 13 features plus 1 target variable.

**Data source:** `Data_set/processed.cleveland.data`  
**Feature count:** 13 clinical attributes + 1 target column  
**Sample size:** 303 instances

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to import custom src modules
ROOT_DIR = os.path.abspath(os.path.join('..'))
sys.path.append(ROOT_DIR)

from src.data_preprocessing import load_data, preprocess_data, summarize_target, split_features_target
from src.logistic_regression_model import (
    scale_train_test,
    train_logistic_regression,
    get_sorted_coefficients
)
from src.evaluation import evaluate_model, plot_confusion_matrix, plot_roc_curve, save_metrics

print('✅ All imports successful')

## 3. Data Preprocessing

In [ ]:
# Load dataset
DATA_PATH = os.path.join('..', 'Data_set', 'processed.cleveland.data')

raw_df = load_data(DATA_PATH)
print(f'Raw dataset shape: {raw_df.shape}')
print(f'\nFirst 5 rows:')
print(raw_df.head())

In [ ]:
print('Dataset Info:')
print(raw_df.info())
print('\n' + '='*50 + '\n')
print('Basic Statistics:')
print(raw_df.describe().round(2))

In [ ]:
# Check for missing values before preprocessing
print('Missing value counts (before preprocessing):')
print(raw_df.isnull().sum())

In [ ]:
# Apply preprocessing: type conversion, missing value imputation, binary target
clean_df = preprocess_data(raw_df)
print(f'Clean dataset shape: {clean_df.shape}')
print(f'\nFirst 5 rows after preprocessing:')
print(clean_df.head())

print('\nMissing values (after preprocessing):')
print(clean_df.isnull().sum().sum(), 'missing values')

### 3.1 Binary Target Variable Conversion

The original dataset uses values 0, 1, 2, 3, 4 to represent:
- 0 = no heart disease
- 1-4 = increasing levels of heart disease severity

For logistic regression binary classification, the target is converted to:
- **0** = no heart disease
- **1** = heart disease present (combines original 1-4 values)

This simplifies the problem and aligns with standard binary classification objectives.

In [ ]:
# Verify binary target distribution
print('Binary target distribution:')
target_counts = summarize_target(clean_df)
print(target_counts)
print(f'\nClass balance: {target_counts[0]/(target_counts[0]+target_counts[1]):.2%} (no disease), {target_counts[1]/(target_counts[0]+target_counts[1]):.2%} (disease)')

## 4. Why Logistic Regression?

Logistic Regression is an ideal baseline classifier for binary diseases:
- **Interpretability**: Each feature's coefficient shows its impact on disease risk
- **Probability output**: Provides disease probability (not just binary prediction)
- **Fast training**: Efficient and suitable for clinical applications
- **Statistical foundation**: Clear probabilistic model with well-studied properties

In [ ]:
# 4. Prepare data for model training
X, y = split_features_target(clean_df)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'Features: {X_train.shape[1]}')

In [ ]:
# Scale features using StandardScaler (fit on training data only)
X_train_scaled, X_test_scaled, scaler = scale_train_test(X_train, X_test)
print('✅ Features scaled with StandardScaler')
print(f'Scaling fit on: {X_train.shape[0]} training samples')
print(f'Training features mean: {X_train_scaled.mean(axis=0).round(4)}')
print(f'Training features std: {X_train_scaled.std(axis=0).round(4)}')

## 5. Model Training

In [ ]:
# Train logistic regression model
model = train_logistic_regression(
    X_train_scaled, 
    y_train, 
    solver='lbfgs', 
    max_iter=1000,
    random_state=42
)
print('✅ Logistic Regression model trained successfully')
print(f'Model classes: {model.classes_}')
print(f'Model converged: {model.n_iter_[0]} iterations')

## 6. Model Evaluation

In [ ]:
# Evaluate model on test set
metrics = evaluate_model(model, X_test_scaled, y_test)

print('='*50)
print('LOGISTIC REGRESSION EVALUATION METRICS')
print('='*50)
print(f"Accuracy:  {metrics['accuracy']:.4f}")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall:    {metrics['recall']:.4f}")
print(f"F1-Score:  {metrics['f1_score']:.4f}")
print(f"ROC AUC:   {metrics['roc_auc']:.4f}")
print('='*50)

In [ ]:
# Detailed classification report
print('\nCLASSIFICATION REPORT:')
print(metrics['classification_report'])

In [ ]:
# Confusion matrix
print('CONFUSION MATRIX:')
cm = metrics['confusion_matrix']
print(f"True Negatives:  {cm[0,0]}")
print(f"False Positives: {cm[0,1]}")
print(f"False Negatives: {cm[1,0]}")
print(f"True Positives:  {cm[1,1]}")

## 7. Feature Importance Analysis

Logistic regression coefficients show the linear relationship between features and disease likelihood. Positive coefficients increase disease probability, while negative coefficients decrease it.

In [ ]:
# Extract and sort coefficients by importance
coef_importance = get_sorted_coefficients(model, X.columns.tolist())

print('TOP 10 FEATURES BY IMPORTANCE (by absolute coefficient):')
print('='*60)
display_df = coef_importance.head(10).copy()
display_df['coefficient'] = display_df['coefficient'].round(4)
display_df['abs_coefficient'] = display_df['abs_coefficient'].round(4)
print(display_df.to_string(index=False))

In [ ]:
# Separate positive and negative coefficients for interpretation
positive_coefs = coef_importance[coef_importance['coefficient'] > 0].head(5)
negative_coefs = coef_importance[coef_importance['coefficient'] < 0].head(5)

print('\n' + '='*60)
print('FEATURES INCREASING DISEASE RISK (Positive Coefficients):')
print('='*60)
for idx, row in positive_coefs.iterrows():
    print(f"{idx+1:2d}. {row['feature']:12s} → {row['coefficient']:+.4f}")

print('\n' + '='*60)
print('FEATURES DECREASING DISEASE RISK (Negative Coefficients):')
print('='*60)
for idx, row in negative_coefs.iterrows():
    print(f"{idx+1:2d}. {row['feature']:12s} → {row['coefficient']:+.4f}")

## 8. Results Visualization

In [ ]:
# Ensure results directory exists
RESULTS_DIR = os.path.join('..', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# Save evaluation metrics
metrics_path = os.path.join(RESULTS_DIR, 'logistic_metrics.txt')
save_metrics(metrics, metrics_path)
print(f'✅ Metrics saved to: {metrics_path}')

# Generate and save confusion matrix
confusion_path = os.path.join(RESULTS_DIR, 'logistic_confusion_matrix.png')
plot_confusion_matrix(metrics['confusion_matrix'], save_path=confusion_path)
print(f'✅ Confusion matrix saved to: {confusion_path}')

# Generate and save ROC curve
roc_path = os.path.join(RESULTS_DIR, 'logistic_roc_curve.png')
y_scores = model.predict_proba(X_test_scaled)[:, 1]
plot_roc_curve(y_test, y_scores, save_path=roc_path)
print(f'✅ ROC curve saved to: {roc_path}')

## 9. Interpretation of Results

**Model Performance:**
- The model achieves reasonable accuracy on the test set, balancing sensitivity and specificity
- The ROC-AUC score indicates discriminative ability between disease/non-disease patients
- Precision and Recall trade-off should be considered based on clinical context

**Feature Interpretation:**
- Features with positive coefficients increase the predicted probability of heart disease
- Features with negative coefficients indicate lower disease risk
- The magnitude of coefficients shows relative feature importance in the linear model

## 10. Critical Analysis and Discussion

**Strengths:**
- Logistic Regression provides a clear baseline model
- Results are highly interpretable for clinical decision support
- Fast training and inference make it suitable for real-world applications
- Probabilistic output enables risk quantification

**Limitations:**
- Model assumes linear relationship between features and log-odds
- Performance depends on proper feature scaling and preprocessing
- Limited to the Cleveland dataset subset (303 samples)
- Binary conversion loses information about disease severity

## 11. Limitations

- **Dataset size**: Only 303 samples from Cleveland source
- **Binary simplification**: Reduces 5-class problem to binary classification
- **Missing value imputation**: Uses median, which may introduce bias
- **Linear assumption**: Logistic regression cannot capture non-linear relationships
- **Class imbalance**: Slightly more negative cases (164) than positive (139)

## 12. Future Improvements

- **Ensemble methods**: Compare against Random Forest, SVM, Gradient Boosting
- **Cross-validation**: Implement k-fold CV for more stable performance estimates
- **Hyperparameter tuning**: Optimize regularization strength (C parameter)
- **Feature engineering**: Explore polynomial features, interaction terms
- **Multi-class approach**: Maintain original 5-class severity levels
- **Imbalance handling**: Apply SMOTE or class weights for better balance

## 13. Individual Contribution (Member 1)

**Logistic Regression Pipeline:**
- ✅ Built modular Python modules for preprocessing, training, and evaluation
- ✅ Created `notebooks/01_logistic_regression_analysis.ipynb` with complete analysis
- ✅ Implemented data loading with proper encoding handling
- ✅ Applied StandardScaler correctly (fit on training data only)
- ✅ Trained logistic regression with appropriate convergence settings
- ✅ Generated evaluation metrics: Accuracy, Precision, Recall, F1, ROC-AUC
- ✅ Created visualizations: Confusion Matrix, ROC Curve
- ✅ Analyzed feature coefficients and disease risk interpretation
- ✅ Saved all results to `results/` directory
- ✅ Documented problem definition, methodology, and limitations

**Files Created:**
- `src/data_preprocessing.py` — Dataset loading, cleaning, type conversion
- `src/logistic_regression_model.py` — Feature scaling and model training
- `src/evaluation.py` — Metrics computation and visualization
- `notebooks/01_logistic_regression_analysis.ipynb` — Complete analysis notebook
- `requirements.txt` — Project dependencies

# Logistic Regression Analysis for Heart Disease Prediction
**Machine Learning Assignment — Member 1**  
**Algorithm: Logistic Regression (Binary Classification)|**


**Machine Learning Assignment — Member 1**  
**Algorithm: Logistic Regression (Binary Classification)**

---


Predict whether a patient has heart disease using the Cleveland dataset. The model will classify patients into:
- **0** = no heart disease
- **1** = heart disease present

The goal is to build a reproducible logistic regression pipeline with clean preprocessing, model training, and evaluation.


The Cleveland dataset contains clinical measurements for heart disease diagnosis. It has 14 columns, including the target value.

**Data source:** `Data_set/processed.cleveland.data`  
**Feature count:** 13 clinical attributes + 1 target column

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to import custom src modules
ROOT_DIR = os.path.abspath(os.path.join('..'))
sys.path.append(ROOT_DIR)

from src.data_preprocessing import load_data, preprocess_data, summarize_target, split_features_target
from src.logistic_regression_model import (
    scale_train_test,
    train_logistic_regression,
    get_sorted_coefficients
)
from src.evaluation import evaluate_model, plot_confusion_matrix, plot_roc_curve, save_metrics

print('✅ Imports complete')

<VSCode.Cell language="python"># Dataset path relative to the notebook location
DATA_PATH = os.path.join('..', 'Data_set', 'processed.cleveland.data')

# Load raw dataset
raw_df = load_data(DATA_PATH)
print(f'Dataset shape: {raw_df.shape}')
raw_df.head()

In [ ]:
print(raw_df.isnull().sum())

print('\nData types before conversion:')
print(raw_df.dtypes)

In [ ]:
clean_df = preprocess_data(raw_df)
print(f'Clean dataset shape: {clean_df.shape}')
clean_df.head()


The original Cleveland target values are `0` (no disease) and `1-4` (disease present). For a standard binary classification task, the target is converted to:
- `0` = no heart disease
- `1` = heart disease present

This simplifies the problem and aligns with the logistic regression objective of modeling disease likelihood.

In [ ]:
print(summarize_target(clean_df))


Logistic Regression is a widely used baseline classification algorithm. It is suitable for binary outcomes, produces interpretable coefficients, and provides probability estimates. In this problem, it helps to identify which clinical features increase or decrease heart disease risk.

In [ ]:
X, y = split_features_target(clean_df)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training samples: {X_train.shape[0]}')
print(f'Test samples: {X_test.shape[0]}')

In [ ]:
X_train_scaled, X_test_scaled, scaler = scale_train_test(X_train, X_test)
print('✅ Feature scaling complete')

<VSCode.Cell language="python">model = train_logistic_regression(X_train_scaled, y_train, solver='lbfgs', max_iter=1000)
print('✅ Logistic Regression model trained')

<VSCode.Cell language="python">feature_importance = get_sorted_coefficients(model, X.columns.tolist())
feature_importance.head(10)


Features are sorted by absolute coefficient value. Positive coefficients increase the predicted probability of heart disease, while negative coefficients decrease it.

<VSCode.Cell language="python">metrics = evaluate_model(model, X_test_scaled, y_test)
print('Accuracy:', round(metrics['accuracy'], 4))
print('Precision:', round(metrics['precision'], 4))
print('Recall:', round(metrics['recall'], 4))
print('F1 Score:', round(metrics['f1_score'], 4))
print('ROC AUC:', round(metrics['roc_auc'], 4))

print('\nClassification Report:')
print(metrics['classification_report'])

In [ ]:
RESULTS_DIR = os.path.join('..', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

metrics_path = os.path.join(RESULTS_DIR, 'logistic_metrics.txt')
confusion_path = os.path.join(RESULTS_DIR, 'logistic_confusion_matrix.png')
roc_path = os.path.join(RESULTS_DIR, 'logistic_roc_curve.png')

save_metrics(metrics, metrics_path)
plot_confusion_matrix(metrics['confusion_matrix'], save_path=confusion_path)
plot_roc_curve(y_test, model.predict_proba(X_test_scaled)[:, 1], save_path=roc_path)

print(f'Saved metrics to: {metrics_path}')
print(f'Saved confusion matrix to: {confusion_path}')
print(f'Saved ROC curve to: {roc_path}')


The logistic regression coefficients show which features are most strongly associated with heart disease risk. Positive coefficients indicate features that increase the predicted probability of disease, while negative coefficients indicate protective or lower-risk attributes.


Logistic Regression provides a clear baseline for this task. The model is easy to interpret and helps identify important clinical indicators. However, performance depends on clean data and correct handling of class imbalance.

The results should be compared against stronger models such as random forest or support vector machines in a broader group analysis.


- The dataset is limited to the Cleveland subset.
- Binary conversion removes information about disease severity.
- Logistic Regression assumes a linear relationship between features and log-odds.
- Missing values are imputed, which may introduce bias.


- Compare logistic regression with ensemble methods.
- Add cross-validation for more stable performance estimates.
- Test additional imputation strategies and feature engineering.
- Explore regularization to reduce overfitting.


- Implemented complete logistic regression pipeline for heart disease prediction.
- Built modular preprocessing, model training, and evaluation utilities in `src/`.
- Added notebook `notebooks/01_logistic_regression_analysis.ipynb` with data loading, cleaning, training, and evaluation.
- Generated evaluation outputs into the `results/` folder, including metrics, confusion matrix, and ROC curve.
- Provided feature importance analysis and a clear interpretation of model results.